In [5]:
%pip install -U langchain-huggingface python-dotenv langgraph


Note: you may need to restart the kernel to use updated packages.


In [21]:
import os
import operator
from typing import TypedDict, List, Annotated
from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.output_parsers import JsonOutputParser
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

# Load your token from .env
load_dotenv()

# Initialize a different model that is free and direct on Hugging Face
llm_endpoint = HuggingFaceEndpoint(
    repo_id="HuggingFaceH4/zephyr-7b-beta", # Or "google/gemma-2-9b-it"
    task="text-generation",
    max_new_tokens=1024,
    do_sample=False,
)
llm = ChatHuggingFace(llm=llm_endpoint)



In [22]:
class Task(BaseModel):
    id: int
    title: str
    brief: str = Field(..., description="What to cover")

In [23]:
class Plan(BaseModel):
    blog_title: str
    tasks: List[Task]

In [24]:
class State(TypedDict):
    topic: str
    plan: Plan
    # reducer: results from workers get concatenated automatically
    sections: Annotated[List[str], operator.add]
    final: str

In [49]:
def orchestrator(state: State) -> dict:
    import json
    import ast
    import re
    
    prompt = (
        "You are a blog planner. Output ONLY a valid JSON object. No extra text.\n"
        f"TOPIC: {state['topic']}\n"
        "Format: {\"blog_title\": \"string\", \"tasks\": [{\"id\": 1, \"title\": \"string\", \"brief\": \"string\"}]}"
    )

    response = llm.invoke([HumanMessage(content=prompt)])
    content = response.content.strip()
    
    # 1. Find the JSON part between { and }
    match = re.search(r'\{.*\}', content, re.DOTALL)
    if not match:
        print("❌ Could not find any brackets in output")
        return {"plan": Plan(blog_title="Default Plan", tasks=[Task(id=1, title="Intro", brief="write intro")])}
    
    clean_content = match.group()

    # 2. Try Strategy A: Standard JSON
    try:
        plan_dict = json.loads(clean_content)
        print("✅ Strategy A (JSON) worked!")
    except:
        # 3. Try Strategy B: Python Literal (Handles single quotes correctly)
        try:
            plan_dict = ast.literal_eval(clean_content)
            print("✅ Strategy B (Python Literal) worked!")
        except Exception as e:
            print(f"❌ Both strategies failed. Error: {e}")
            return {"plan": Plan(blog_title="Fallback", tasks=[Task(id=1, title="Intro", brief="write intro")])}

    plan = Plan(**plan_dict)
    return {"plan": plan}


In [50]:
def fanout(state: State):
    return [Send("worker", {"task": task, "topic": state["topic"], "plan": state["plan"]})
            for task in state["plan"].tasks]

In [51]:
def worker(payload: dict) -> dict:

    # payload contains what we sent
    task = payload["task"]
    topic = payload["topic"]
    plan = payload["plan"]

    blog_title = plan.blog_title

    section_md = llm.invoke(
        [
            SystemMessage(content="Write one clean Markdown section."),
            HumanMessage(
                content=(
                    f"Blog: {blog_title}\n"
                    f"Topic: {topic}\n\n"
                    f"Section: {task.title}\n"
                    f"Brief: {task.brief}\n\n"
                    "Return only the section content in Markdown."
                )
            ),
        ]
    ).content.strip()

    return {"sections": [section_md]}

In [52]:
from pathlib import Path

def reducer(state: State) -> dict:
    
    title = state["plan"].blog_title
    body = "\n\n".join(state["sections"]).strip()

    final_md = f"# {title}\n\n{body}\n"

    # ---- save to file ----
    filename = title.lower().replace(" ", "_") + ".md"
    output_path = Path(filename)
    output_path.write_text(final_md, encoding="utf-8")

    return {"final": final_md}


In [53]:
# 1. Define the Graph
g = StateGraph(State)

# 2. Add your nodes
g.add_node("orchestrator", orchestrator)
g.add_node("worker", worker)
g.add_node("reducer", reducer)

# 3. Add your edges
g.add_edge(START, "orchestrator")
g.add_conditional_edges("orchestrator", fanout, ["worker"])
g.add_edge("worker", "reducer")
g.add_edge("reducer", END)

# 4. COMPILE (This creates the 'app' variable)
app = g.compile()


In [54]:
out = app.invoke({"topic": "Write a blog on Self Attention", "sections": []})

✅ Strategy A (JSON) worked!
